# Task 3 — RAG Core Logic & Qualitative Evaluation
**Project:** CrediTrust Complaint RAG Chatbot

This notebook:
1. Loads the full-dataset FAISS index built from `complaint_embeddings.parquet`
2. Wires up the retriever, prompt template, and Hugging Face Inference API generator (`src/`)
3. Runs the pipeline against 8 representative questions
4. Builds the Markdown evaluation table required for the report

**Before running:** set your Hugging Face token as an environment variable
(see `src/generator.py` for OS-specific instructions), and make sure you've
already run `src/build_faiss_index.py` so `vector_store/full_dataset.faiss`
and `vector_store/full_dataset_metadata.parquet` exist.


In [1]:
import sys
sys.path.insert(0, "../src")

import os
import pandas as pd

from vector_index import load_index_and_metadata
from embedding import Embedder
from retriever import Retriever
from generator import Generator
from rag_pipeline import RAGPipeline

pd.set_option("display.max_colwidth", 200)


In [2]:
INDEX_PATH = "../vector_store/full_dataset.faiss"
METADATA_PATH = "../vector_store/full_dataset_metadata.parquet"

index, metadata_df = load_index_and_metadata(INDEX_PATH, METADATA_PATH)
print(f"Loaded index with {index.ntotal:,} chunks.")


Loaded index with 1,375,327 chunks.


In [3]:
embedder = Embedder()  # sentence-transformers/all-MiniLM-L6-v2, same as Task 2
retriever = Retriever(index, metadata_df, embedder)

# Requires HF_TOKEN to be set as an environment variable.
generator = Generator(model_name="deepseek-ai/DeepSeek-V3-0324")

pipeline = RAGPipeline(retriever, generator, k=5)
print("Pipeline ready.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pipeline ready.


## Test questions

Eight questions spanning all four products plus a couple of cross-product
themes (fraud, customer service) that should surface complaints from
multiple categories at once.

In [4]:
TEST_QUESTIONS = [
    "Why are people unhappy with Credit Cards?",
    "What are the most common complaints about Personal Loans?",
    "What issues do customers report with Savings Accounts?",
    "What problems are customers experiencing with Money Transfers?",
    "Are there complaints about unauthorized transactions?",
    "Do customers complain about poor customer service?",
    "What billing or fee-related issues appear across products?",
    "Are there any complaints related to fraud?",
]
len(TEST_QUESTIONS)


8

In [5]:
def truncate(text, n=160):
    text = str(text).replace("\n", " ").strip()
    return text if len(text) <= n else text[:n].rstrip() + "..."


eval_rows = []
for question in TEST_QUESTIONS:
    result = pipeline.answer(question, k=5)
    top_sources = result["sources"][:2]
    sources_str = " | ".join(
        f"[{s['product_category']}] {truncate(s['chunk_text'], 120)}" for s in top_sources
    )
    eval_rows.append({
        "Question": question,
        "Generated Answer": result["answer"],
        "Retrieved Sources (top 2)": sources_str,
        "Quality Score (1-5)": None,   # fill in manually after reading the answer
        "Comments/Analysis": "",        # fill in manually after reading the answer
    })
    print(f"Done: {question}")

eval_df = pd.DataFrame(eval_rows)
eval_df


Done: Why are people unhappy with Credit Cards?
Done: What are the most common complaints about Personal Loans?
Done: What issues do customers report with Savings Accounts?
Done: What problems are customers experiencing with Money Transfers?
Done: Are there complaints about unauthorized transactions?
Done: Do customers complain about poor customer service?
Done: What billing or fee-related issues appear across products?
Done: Are there any complaints related to fraud?


,Question,Generated Answer,Retrieved Sources (top 2),Quality Score (1-5),Comments/Analysis
0,Why are people unhappy with Credit Cards?,"Based on the retrieved complaint excerpts, people are unhappy with credit cards for several reasons: \n\n1. **Perceived Unfair Denials** – Some customers feel unfairly denied credit despite havin...","[Credit Card] uspect that this is ageism. i am not at all happy. i honestly do not need the credit card, but am offended at the denial... | [Credit Card] pitalism favors the corporations and billi...",None,
1,What are the most common complaints about Personal Loans?,"Based on the retrieved complaint excerpts, the most common complaints about **Personal Loans** include: \n\n1. **Harassment by Employees**: Customers report being constantly harassed by employees...","[Personal Loan] constantly being harassed by employees about a loan that is not mine! | [Savings Account] the ones issueing the loan, absolutely ridiculous. i needed this loan because of the econo...",None,
2,What issues do customers report with Savings Accounts?,"Based on the retrieved complaint excerpts, customers report several issues with **Savings Accounts**, including: \n\n1. **Unauthorized Account Activity**: Customers experienced unexpected and out...",[Savings Account] h detail as i feel is relevant for this complaint. all of these calls should be recorded and available for review to pro... | [Savings Account] ing account every month to keep a ...,None,
3,What problems are customers experiencing with Money Transfers?,"Based on the retrieved complaint excerpts, customers are experiencing several significant problems with money transfers, including: \n\n1. **Operational Issues**: Problems with sending and receiv...","[Money Transfer] problems with money transfer, receiving and sending out. | [Money Transfer] les and money transfers rather than passing the entire burden to the customers to do that. moreover the...",None,
4,Are there complaints about unauthorized transactions?,"Yes, there are multiple complaints about unauthorized transactions across the provided sources. Here’s a breakdown of the complaints: \n\n1. **Source 1**: The customer expresses frustration over ...",[Money Transfer] i am writing to formally express my frustration and dissatisfaction with the handling of unauthorized transactions proce... | [Money Transfer] s and practices for handling unautho...,None,
5,Do customers complain about poor customer service?,"Yes, customers frequently complain about poor customer service across multiple sources. \n\nHere’s the breakdown: \n- **Credit Card**: Multiple complaints mention poor customer service (Sources ...",[Credit Card] the poor customer service. | [Credit Card] n about poor customer service!,None,
6,What billing or fee-related issues appear across products?,"The retrieved complaint excerpts highlight several billing and fee-related issues across credit cards and savings accounts: \n\n1. **Unfair or Excessive Fees** – Customers report unexpected fees,...","[Credit Card] practice which is unfair and harmful to consumers. the best description of my issue includes billing statements, late fe... | [Credit Card] did n't seem to review or follow-up on any...",None,
7,Are there any complaints related to fraud?,"Yes, there are complaints related to fraud in the provided context. Here are the relevant excerpts: \n\n1. **Source 1 (Personal Loan):** The customer mentions reporting a fraud claim and threaten...",[Personal Loan] stently since first reporting the fraud claim on . i am now forced to submit formal complaints through the regulatory ag... | [Money Transfer] frauding customers?,None,


## Review each answer

For each row above:
1. Read the generated answer against its retrieved sources — does the answer
   actually reflect what the sources say, or does it overreach/hallucinate?
2. Score 1 (poor) to 5 (excellent) in `Quality Score (1-5)`.
3. Add a short note in `Comments/Analysis` — e.g. *"Answer correctly
   summarizes the two sources, but only retrieved Credit Card chunks even
   though Savings Accounts also have similar complaints"* or *"Hallucinated
   a dollar amount not present in any retrieved source."*

Edit the cell below directly with your scores once you've reviewed the
table above (replace the `None`/empty values).

In [8]:
# Fill these in after manually reviewing the answers above, in the same
# order as TEST_QUESTIONS.
quality_scores = [4, 5, 3, 4, 4, 3, 4, 1]
comments = ["", "", "", "", "", "", "", ""]

eval_df["Quality Score (1-5)"] = quality_scores
eval_df["Comments/Analysis"] = comments
eval_df


,Question,Generated Answer,Retrieved Sources (top 2),Quality Score (1-5),Comments/Analysis
0,Why are people unhappy with Credit Cards?,"Based on the retrieved complaint excerpts, people are unhappy with credit cards for several reasons: \n\n1. **Perceived Unfair Denials** – Some customers feel unfairly denied credit despite havin...","[Credit Card] uspect that this is ageism. i am not at all happy. i honestly do not need the credit card, but am offended at the denial... | [Credit Card] pitalism favors the corporations and billi...",4,
1,What are the most common complaints about Personal Loans?,"Based on the retrieved complaint excerpts, the most common complaints about **Personal Loans** include: \n\n1. **Harassment by Employees**: Customers report being constantly harassed by employees...","[Personal Loan] constantly being harassed by employees about a loan that is not mine! | [Savings Account] the ones issueing the loan, absolutely ridiculous. i needed this loan because of the econo...",5,
2,What issues do customers report with Savings Accounts?,"Based on the retrieved complaint excerpts, customers report several issues with **Savings Accounts**, including: \n\n1. **Unauthorized Account Activity**: Customers experienced unexpected and out...",[Savings Account] h detail as i feel is relevant for this complaint. all of these calls should be recorded and available for review to pro... | [Savings Account] ing account every month to keep a ...,3,
3,What problems are customers experiencing with Money Transfers?,"Based on the retrieved complaint excerpts, customers are experiencing several significant problems with money transfers, including: \n\n1. **Operational Issues**: Problems with sending and receiv...","[Money Transfer] problems with money transfer, receiving and sending out. | [Money Transfer] les and money transfers rather than passing the entire burden to the customers to do that. moreover the...",4,
4,Are there complaints about unauthorized transactions?,"Yes, there are multiple complaints about unauthorized transactions across the provided sources. Here’s a breakdown of the complaints: \n\n1. **Source 1**: The customer expresses frustration over ...",[Money Transfer] i am writing to formally express my frustration and dissatisfaction with the handling of unauthorized transactions proce... | [Money Transfer] s and practices for handling unautho...,4,
5,Do customers complain about poor customer service?,"Yes, customers frequently complain about poor customer service across multiple sources. \n\nHere’s the breakdown: \n- **Credit Card**: Multiple complaints mention poor customer service (Sources ...",[Credit Card] the poor customer service. | [Credit Card] n about poor customer service!,3,
6,What billing or fee-related issues appear across products?,"The retrieved complaint excerpts highlight several billing and fee-related issues across credit cards and savings accounts: \n\n1. **Unfair or Excessive Fees** – Customers report unexpected fees,...","[Credit Card] practice which is unfair and harmful to consumers. the best description of my issue includes billing statements, late fe... | [Credit Card] did n't seem to review or follow-up on any...",4,
7,Are there any complaints related to fraud?,"Yes, there are complaints related to fraud in the provided context. Here are the relevant excerpts: \n\n1. **Source 1 (Personal Loan):** The customer mentions reporting a fraud claim and threaten...",[Personal Loan] stently since first reporting the fraud claim on . i am now forced to submit formal complaints through the regulatory ag... | [Money Transfer] frauding customers?,1,


## Export the Markdown table for the report

In [7]:
markdown_table = eval_df.to_markdown(index=False)
print(markdown_table)

with open("../reports/task3_evaluation_table.md", "w") as f:
    f.write("# Task 3 — Qualitative Evaluation\n\n")
    f.write(markdown_table)
    f.write("\n")

print("\nSaved to reports/task3_evaluation_table.md")


| Question                                                       | Generated Answer                                                                                                                                                                                                                                                              | Retrieved Sources (top 2)                                                                                                                                                                                                                                                                     | Quality Score (1-5)   | Comments/Analysis   |
|:---------------------------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Summary observations

After filling in the scores above, write 2-3 sentences here (for your own
reference, and to paste into the final report) on overall patterns: Did the
retriever consistently pull relevant chunks? Did the LLM stick to the
provided context, or did it occasionally answer from general knowledge?
Were any product categories harder to get good answers for than others?